In [1]:
import pandas as pd
import numpy as np

In [2]:
df_emailData = pd.read_csv('../DataSets_/mergeSet.csv', encoding='latin1')

In [3]:
df_emailData.columns

Index(['Unnamed: 0', 'subject', 'body', 'queue', 'priority', 'language'], dtype='object')

In [4]:
df_emailData.drop(columns=["Unnamed: 0"], inplace=True)


In [5]:
df_emailData.dropna(inplace=True)

In [6]:
df_emailData.head()

,subject,body,queue,priority,language
0,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",Technical Support,high,en
1,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Returns and Exchanges,medium,en
2,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Billing and Payments,low,en
3,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Sales and Pre-Sales,medium,en
4,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Technical Support,high,en


In [7]:
df_emailData["combined_text"] = df_emailData["subject"] + " " + df_emailData["body"]

In [8]:
df_emailData['queue'].value_counts()

queue
Technical Support                  7090
Product Support                    4643
Customer Service                   3755
IT Support                         2883
Billing and Payments               2543
Returns and Exchanges              1216
Service Outages and Maintenance     996
Sales and Pre-Sales                 692
Human Resources                     468
General Inquiry                     335
Name: count, dtype: int64

In [9]:
import re
import string

In [10]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df_emailData["clean_text"] = df_emailData["combined_text"].apply(clean_text)

In [11]:
from sklearn.utils import resample
import pandas as pd

In [12]:
df_hybrid = df_emailData.copy()

In [13]:
lower_target = 2500
upper_target = 5090
balanced_list = []

for label in df_hybrid["queue"].unique():
    
    df_class = df_hybrid[df_hybrid["queue"] == label]
    class_size = len(df_class)
     # Oversample small classes
    if class_size < lower_target:
        df_balanced = resample(
            df_class,
            replace=True,
            n_samples=lower_target,
            random_state=42
        )
         # Undersample very large classes
    elif class_size > upper_target:
        df_balanced = resample(
            df_class,
            replace=False,
            n_samples=upper_target,
            random_state=42
        )

        # Keep medium classes unchanged
    else:
        df_balanced = df_class
    
    balanced_list.append(df_balanced)

df_final = pd.concat(balanced_list)
df_final = df_final.sample(frac=1, random_state=42)

print(df_final["queue"].value_counts())

queue
Technical Support                  5090
Product Support                    4643
Customer Service                   3755
IT Support                         2883
Billing and Payments               2543
Returns and Exchanges              2500
Sales and Pre-Sales                2500
General Inquiry                    2500
Service Outages and Maintenance    2500
Human Resources                    2500
Name: count, dtype: int64


In [14]:
df_final["combined_text"] = df_final["subject"] + " " + df_final["body"]
df_final["clean_text"] = df_final["combined_text"].apply(clean_text)


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [16]:
tfidf = TfidfVectorizer(
    max_features=60000,
    ngram_range=(1,3),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True,
    stop_words="english"
)

In [17]:
tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3,5),
    max_features=10000
)

In [18]:
X_word = tfidf.fit_transform(df_final["clean_text"])
X_char = tfidf_char.fit_transform(df_final["clean_text"])


In [19]:
from scipy.sparse import hstack

In [20]:
X = hstack([X_word, X_char])
y = df_final["queue"]


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [22]:
from sklearn.svm import LinearSVC
model = LinearSVC(class_weight="balanced")
model.fit(X_train, y_train)

LinearSVC(class_weight='balanced')

In [23]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

                                 precision    recall  f1-score   support

           Billing and Payments       0.89      0.87      0.88       508
               Customer Service       0.70      0.70      0.70       751
                General Inquiry       0.99      1.00      1.00       500
                Human Resources       0.97      1.00      0.98       500
                     IT Support       0.68      0.68      0.68       577
                Product Support       0.71      0.67      0.69       929
          Returns and Exchanges       0.88      0.94      0.91       500
            Sales and Pre-Sales       0.96      0.99      0.97       500
Service Outages and Maintenance       0.93      0.98      0.95       500
              Technical Support       0.69      0.67      0.68      1018

                       accuracy                           0.81      6283
                      macro avg       0.84      0.85      0.84      6283
                   weighted avg       0.81      0

In [24]:
import pickle

# Save model
with open("category_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save both vectorizers
with open("category_tfidf_word.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("category_tfidf_char.pkl", "wb") as f:
    pickle.dump(tfidf_char, f)

print("Category model and both vectorizers saved successfully!")

Category model and both vectorizers saved successfully!
